# Week 6 - Day 1 (Second Half)
## Task 2 - Use Cases Mini-Lab (+ Bonus)

In [ ]:
# !pip install numpy scikit-learn sentence-transformers matplotlib
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

# Task 2 - Use Cases Mini-Lab
Embedding model: `all-MiniLM-L6-v2` (local, free - same one used in the first half).

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return model.encode(texts, normalize_embeddings=True)  # unit vectors -> dot == cosine

def top_k(query_vec, matrix, k=3):
    sims = matrix @ query_vec
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

## 2.1 Semantic Search (15 FAQs, 3 queries, top 3 each)

In [ ]:
faqs = [
    "How do I reset my password?",
    "How can I track my order?",
    "What is your return policy?",
    "How long does shipping take?",
    "Do you ship internationally?",
    "How do I cancel my subscription?",
    "What payment methods do you accept?",
    "How do I update my billing information?",
    "Can I change my delivery address after ordering?",
    "How do I contact customer support?",
    "Is my personal data secure?",
    "Do you offer discounts for students?",
    "How do I delete my account?",
    "What should I do if my item arrives damaged?",
    "How do I apply a promo code at checkout?",
]
queries = [
    "I forgot my login credentials",
    "where is my package right now",
    "can I get my money back for something I bought",
]

faq_vecs = embed(faqs)
for q in queries:
    print(f"\nQuery: {q}")
    for rank, (i, s) in enumerate(top_k(embed([q])[0], faq_vecs, 3), 1):
        print(f"  {rank}. [{s:.3f}] {faqs[i]}")

**Business use:** a help-centre search that finds the right FAQ even when customers don't use the exact words, cutting support tickets.

## 2.2 Duplicate Detection (12 complaints)

In [ ]:
complaints = [
    "My order arrived two weeks late and nobody told me.",                 # 0  late delivery
    "The package was delivered 14 days after the promised date.",          # 1  late delivery
    "I was charged twice for the same purchase on my credit card.",        # 2  double charge
    "My card got billed two times for one order.",                         # 3  double charge
    "The mobile app crashes every time I open it.",                        # 4  app crash
    "App keeps crashing as soon as I launch it on my phone.",              # 5  app crash
    "Customer support never replied to my email.",                         # 6  no support reply
    "I emailed support three times and got no response.",                  # 7  no support reply
    "The product I received was damaged and broken.",                      # 8  damaged
    "Item arrived with a cracked screen.",                                 # 9  damaged
    "I want to change the delivery address on my account.",                # 10 unique
    "The website is too slow when I try to check out.",                    # 11 unique
]
comp_vecs = embed(complaints)
sim_matrix = comp_vecs @ comp_vecs.T

THRESHOLD = 0.60   # chosen after looking at the score distribution (see bonus for evaluation)
print(f"Threshold chosen: {THRESHOLD}\n")
pairs = [(i, j, sim_matrix[i, j]) for i in range(len(complaints)) for j in range(i + 1, len(complaints))]
found = sorted([p for p in pairs if p[2] >= THRESHOLD], key=lambda x: -x[2])
for i, j, s in found:
    print(f"[{s:.3f}] ({i}) {complaints[i]}\n         ({j}) {complaints[j]}\n")
print(f"{len(found)} near-duplicate pairs found out of {len(pairs)} pairs")

**Business use:** merge repeated complaints/tickets automatically so agents solve one issue once and the team sees the real volume of each problem.

## 2.3 Clustering (30 headlines, K-Means)

In [ ]:
from sklearn.cluster import KMeans

headlines = [
    # sports
    "Local team wins championship after dramatic overtime",
    "Star striker signs record transfer deal with European club",
    "Tennis champion advances to the final after five-set thriller",
    "Olympic sprinter breaks national record in 100 metres",
    "Cricket captain scores century in series decider",
    "Coach resigns after a string of league defeats",
    # technology
    "New smartphone launches with faster chip and better camera",
    "Tech giant unveils AI assistant for everyday tasks",
    "Startup raises funding to build quantum computing hardware",
    "Major software update fixes critical security vulnerability",
    "Chipmaker announces next generation processor for laptops",
    "Researchers release open-source language model",
    # finance
    "Stock markets rally as inflation cools",
    "Central bank holds interest rates steady",
    "Oil prices fall amid weaker global demand",
    "Bank reports record quarterly profit",
    "Rupee strengthens against the dollar",
    "Gold hits all-time high as investors seek safety",
    # health
    "Study links regular exercise to lower risk of heart disease",
    "New vaccine shows strong results in clinical trials",
    "Doctors warn about rising cases of seasonal flu",
    "Hospital opens new cancer treatment centre",
    "Researchers discover link between sleep and memory",
    "Health ministry launches nationwide nutrition campaign",
    # politics / world
    "Parliament passes new education bill after heated debate",
    "Prime minister meets foreign leaders at climate summit",
    "Voters head to the polls in state elections",
    "Government announces new infrastructure spending plan",
    "Opposition parties demand inquiry into policy decision",
    "Trade agreement signed between two neighbouring countries",
]
head_vecs = embed(headlines)

K = 5
km = KMeans(n_clusters=K, random_state=42, n_init=10).fit(head_vecs)

# Name each cluster: compare its centroid with embeddings of candidate topic names
topic_names = ["Sports", "Technology", "Finance & Markets", "Health & Medicine", "Politics & Government"]
topic_vecs = embed(topic_names)
centroids = km.cluster_centers_ / np.linalg.norm(km.cluster_centers_, axis=1, keepdims=True)
cluster_name = {c: topic_names[int(np.argmax(topic_vecs @ centroids[c]))] for c in range(K)}

for c in range(K):
    print(f"\nCluster {c}: {cluster_name[c]}")
    for h, lab in zip(headlines, km.labels_):
        if lab == c:
            print("   -", h)

In [ ]:
# Optional: 2D view of the clusters
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

xy = PCA(n_components=2, random_state=42).fit_transform(head_vecs)
plt.figure(figsize=(8, 6))
for c in range(K):
    m = km.labels_ == c
    plt.scatter(xy[m, 0], xy[m, 1], label=cluster_name[c])
plt.legend(); plt.title("Headline clusters (PCA of embeddings)"); plt.show()

**Business use:** automatically group news/feedback/support content into topics so editors or product teams can see themes without reading everything.

## 2.4 Recommendation (15 products, top 3 similar)

In [ ]:
products = [
    "Wireless noise-cancelling over-ear headphones with 30-hour battery",
    "True wireless earbuds with charging case and water resistance",
    "Bluetooth portable speaker with deep bass and waterproof design",
    "Mechanical gaming keyboard with RGB backlight",
    "Ergonomic wireless mouse for office work",
    "27-inch 4K monitor with HDR for creative professionals",
    "Lightweight laptop with 16GB RAM and all-day battery",
    "Smartwatch with heart-rate monitor and GPS tracking",
    "Fitness tracker band with sleep monitoring",
    "Non-stick frying pan set for everyday cooking",
    "Stainless steel electric kettle with auto shut-off",
    "Programmable coffee maker with built-in grinder",
    "Yoga mat with extra cushioning and non-slip surface",
    "Adjustable dumbbell set for home workouts",
    "Running shoes with breathable mesh and responsive cushioning",
]
prod_vecs = embed(products)

query_product = "Compact earphones with Bluetooth, great sound and sweat-proof for the gym"
print("Given product:", query_product, "\n")
for rank, (i, s) in enumerate(top_k(embed([query_product])[0], prod_vecs, 3), 1):
    print(f"  {rank}. [{s:.3f}] {products[i]}")

**Business use:** a "you may also like" widget on product pages that raises basket size and keeps shoppers browsing.

### One-line business use summary
| Use case | Business use |
|---|---|
| Semantic search | Help-centre search that understands intent, so fewer support tickets. |
| Duplicate detection | Merge repeated tickets/complaints so each issue is solved once. |
| Clustering | Auto-group content or feedback into themes with no manual tagging. |
| Recommendation | "Similar items" suggestions that increase sales. |

---
# Bonus - Precision & Recall of the Duplicate Threshold
Labelled sample: the 12 complaints with a hand-assigned group id; two complaints are a true duplicate pair if they share a group id.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

group_id = [0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 6]   # hand labels; 5 and 6 are unique
y_true, y_score = [], []
for i in range(len(complaints)):
    for j in range(i + 1, len(complaints)):
        y_true.append(int(group_id[i] == group_id[j]))
        y_score.append(sim_matrix[i, j])
y_true, y_score = np.array(y_true), np.array(y_score)
print(f"{len(y_true)} pairs, {y_true.sum()} true duplicate pairs\n")

print(f"{'thr':>5} {'precision':>10} {'recall':>8} {'F1':>6}")
best = (0, None)
for t in np.arange(0.30, 0.91, 0.05):
    y_pred = (y_score >= t).astype(int)
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)
    print(f"{t:5.2f} {p:10.2f} {r:8.2f} {f:6.2f}")
    if f > best[0]:
        best = (f, round(float(t), 2))
print(f"\nBest F1 = {best[0]:.2f} at threshold {best[1]}")

**Reading the table:** a low threshold catches every duplicate (high recall) but flags unrelated pairs (low precision); a high threshold is precise but misses paraphrases (low recall). Pick the threshold with the best F1, or favour precision if wrongly merging tickets is costly, or recall if missing duplicates is costly. Update `THRESHOLD` in 2.2 to match.